In [1]:
pip install mysql-connector-python

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/17.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/17.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/17.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/17.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/17.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/17.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/17.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/17.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/17.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/17.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/17.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/17.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/17.7 MB ? eta -:--:--
   ---------

In [2]:
import pandas as pd
import mysql.connector
from datetime import datetime

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
connection = mysql.connector.connect(
    host="localhost",
    user="root",
    password="YOUR_PASSWORD",
    database="project"
)
cursor = connection.cursor(dictionary=True)

print("MySQL database connected successfully!")

MySQL database connected successfully!


In [5]:
# TAKE USER ID INPUT

print("==============================================")
print("             USER LOGIN DETAILS")
print("==============================================")

user_id = int(input("User ID: "))

 #CHECK WHETHER USER EXISTS

query = """
SELECT *
FROM users
WHERE user_id = %s
"""

cursor.execute(query, (user_id,))
user = cursor.fetchone()

if user is None:
    print("\nUser ID does not exist.")

    cursor.close()
    connection.close()

    raise SystemExit

 # TAKE LOCATION AND DEVICE NAME

location = input("Location: ").strip()
device_name = input("Device Name: ").strip()

current_time = datetime.now()

# Display only login date and time
print("Login Date & Time:", current_time)

 #GET USER INFORMATION

print("\nUser found successfully!")
print("---------------------------------")
print("User Name :", user["user_name"])
print("Department:", user["department"])
print("Role      :", user["role"])

 # CHECK WHETHER DEVICE EXISTS

query = """
SELECT *
FROM devices
WHERE device_name = %s
"""

cursor.execute(query, (device_name,))
device = cursor.fetchone()

 # HANDLE NEW DEVICE

if device is None:
    print("\nNew Device Detected!")

    # Ask for additional information
    device_type = input("Enter Device Type: ").strip()
    operating_system = input("Enter Operating System: ").strip()

    first_seen_date = current_time.date()

    # Insert new device into devices table
    insert_device = """
    INSERT INTO devices
    (
        device_name,
        device_type,
        operating_system,
        first_seen_date
    )
    VALUES
    (
        %s,
        %s,
        %s,
        %s
    )
    """

    cursor.execute(
        insert_device,
        (
            device_name,
            device_type,
            operating_system,
            first_seen_date
        )
    )

    connection.commit()

    # Get automatically generated device_id
    device_id = cursor.lastrowid

    # Since this device is newly created,it is also new for this user.
    new_device = True
    print("New device information saved successfully.")

 # HANDLE EXISTING DEVICE

else:

    device_id = device["device_id"]
    device_type = device["device_type"]
    operating_system = device["operating_system"]

    print("\nDevice found successfully!")
    print("---------------------------------")
    print("Device Type      :", device_type)


 #GET USER'S LOGIN HISTORY
   
    query = """
    SELECT
        lh.login_id,
        lh.login_datetime,
        lh.location,
        lh.login_status,
        lh.failure_reason,
        lh.user_id,
        lh.device_id,
        d.device_name,
        d.device_type,
        d.operating_system
    FROM login_history lh
    LEFT JOIN devices d
        ON lh.device_id = d.device_id
    WHERE lh.user_id = %s
    ORDER BY lh.login_datetime DESC
    """

    cursor.execute(query, (user_id,))
    records = cursor.fetchall()

    # Convert SQL result into Pandas DataFrame
    history_df = pd.DataFrame(records)


# CHECK WHETHER DEVICE IS NEW FOR THIS USER
    
    if history_df.empty:
        new_device = True

    else:
        known_devices = ( history_df["device_id"].dropna().unique())

        if device_id in known_devices:
            new_device = False
        else:
            new_device = True


# GET USER LOGIN HISTORY

# We need history for both new and existing devices.
# If a new device was inserted above, retrieve history now.

query = """
SELECT
    lh.login_id,
    lh.login_datetime,
    lh.location,
    lh.login_status,
    lh.failure_reason,
    lh.user_id,
    lh.device_id,
    d.device_name,
    d.device_type,
    d.operating_system
FROM login_history lh
LEFT JOIN devices d
    ON lh.device_id = d.device_id
WHERE lh.user_id = %s
ORDER BY lh.login_datetime DESC
"""

cursor.execute(query, (user_id,))
records = cursor.fetchall()

# Convert login history into Pandas DataFrame
history_df = pd.DataFrame(records)

# #FAILED LOGIN ANALYSIS

if history_df.empty:
    failed_count = 0
    recent_failed = 0

else:
    history_df["login_datetime"] = pd.to_datetime(
        history_df["login_datetime"]
    )

    # Total previous failed logins
    failed_count = (
        history_df["login_status"] == "Failed"
    ).sum()

    # Last 24 hours
    last_24_hours = current_time - pd.Timedelta(hours=24)

    recent_failed = (
        (history_df["login_status"] == "Failed") &
        (history_df["login_datetime"] >= last_24_hours)
    ).sum()


# CHECK WHETHER LOCATION IS NEW

if history_df.empty:
    new_location = True

else:
    known_locations = (
        history_df["location"]
        .dropna()
        .unique()
    )

    if location in known_locations:
        new_location = False

    else:
        new_location = True

# CHECK UNUSUAL LOGIN TIME

current_hour = current_time.hour

if current_hour >= 22 or current_hour < 6:
    unusual_time = True

else:
    unusual_time = False


# START RISK SCORE

risk_score = 0
reasons = []

if new_location:
    risk_score += 25
    reasons.append("Login from a new location")

if new_device:
    risk_score += 25
    reasons.append("Login from a new device")

if unusual_time:
    risk_score += 20
    reasons.append("Login occurred during unusual hours")

if recent_failed >= 3:
    risk_score += 20
    reasons.append("Multiple failed login attempts in the last 24 hours")

elif recent_failed > 0:
    risk_score += 10
    reasons.append("Failed login attempts detected in the last 24 hours")

# HIGH NUMBER OF PREVIOUS FAILURES → +10

if failed_count >= 5:
    risk_score += 10
    reasons.append("High number of previous failed login attempts")

# KEEP RISK SCORE BETWEEN 0 AND 100

risk_score = max(0, min(risk_score, 100))

# DETERMINE RISK LEVEL

if risk_score <= 30:
    risk_level = "Low"
elif risk_score <= 60:
    risk_level = "Medium"
else:
    risk_level = "High"


# GENERATE REASON

if len(reasons) == 0:
    reason = "Normal login behavior detected."

else:
    reason = ", ".join(reasons)


# GENERATE RECOMMENDATION

if risk_level == "Low":
    recommendation = ("Allow login and continue normal monitoring.")

elif risk_level == "Medium":
    recommendation = ("Allow login but verify the user's activity.")
    
else:
    recommendation = ("Verify user identity and investigate the login activity.")

# FINAL CYBERSECURITY RISK RESULT

print("\n")
print("====================================================")
print("             CYBERSECURITY RISK RESULT")
print("====================================================")

print("Risk Score       :", risk_score)
print("Risk Level       :", risk_level)
print("Reason           :", reason)
print("Recommendation   :", recommendation)

print("====================================================")


# INSERT LOGIN INTO login_history

insert_login = """
INSERT INTO login_history
(
    login_datetime,
    location,
    login_status,
    failure_reason,
    user_id,
    device_id
)
VALUES
(
    %s,
    %s,
    %s,
    %s,
    %s,
    %s
)
"""

cursor.execute(
    insert_login,
    (
        current_time,
        location,
        "Success",
        None,
        user_id,
        device_id
    )
)

connection.commit()

login_id = cursor.lastrowid

# INSERT RISK RESULT INTO risk_result

insert_risk = """
INSERT INTO risk_result
(
    risk_score,
    risk_level,
    reason,
    recommendation,
    calculated_at,
    user_id,
    login_id
)
VALUES
(
    %s,
    %s,
    %s,
    %s,
    %s,
    %s,
    %s
)
"""

cursor.execute(
    insert_risk,
    (
        risk_score,
        risk_level,
        reason,
        recommendation,
        current_time,
        user_id,
        login_id
    )
)

connection.commit()

# DATABASE CONFIRMATION

print("\nLogin history and risk result saved in MySQL.")

# CLOSE CONNECTION

cursor.close()
connection.close()

print("Database connection closed.")
print("Program completed successfully!")

             USER LOGIN DETAILS


User ID:  101
Location:  Pune
Device Name:  LAP-102


Login Date & Time: 2026-10-07 20:27:09.373646

User found successfully!
---------------------------------
User Name : Rahul Sharma
Department: Finance
Role      : Analyst

Device found successfully!
---------------------------------
Device Type      : Laptop


             CYBERSECURITY RISK RESULT
Risk Score       : 25
Risk Level       : Low
Reason           : Login from a new device
Recommendation   : Allow login and continue normal monitoring.

Login history and risk result saved in MySQL.
